<a href="https://colab.research.google.com/github/meghansn/mental-health-llm-pipeline/blob/main/Clean_PHI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install transformers torch google-cloud-bigquery pandas

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
#initalize the hugging face pipeline
from transformers import pipeline
import torch

print("⏳ Loading local Hugging Face NER pipeline with sliding window...")

# Check if a GPU is available to make it run faster, otherwise use CPU
device = 0 if torch.cuda.is_available() else -1

ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    stride=64,           # Sliders overlapping windows to read long text
    device=device        # Automatically uses GPU if Colab gave you one
)
print("✅ Model loaded with stride capabilities.")

In [ ]:
# 1. Define your data generation constants first
GTA_CITIES = ["Toronto", "Mississauga", "Brampton", "Markham", "Vaughan", "Richmond Hill", "Oakville", "Burlington"]

PROFILES = {
    "Depressive": {"symptoms": ["persistent sadness", "fatigue", "lack of interest"], "meds": [("Sertraline", "769747"), ("Escitalopram", "237646")]},
    "Anxiety": {"symptoms": ["excessive worry", "restlessness", "muscle tension"], "meds": [("Buspirone", "313498"), ("Lorazepam", "38634")]},
    "Bipolar": {"symptoms": ["mood swings", "impulsivity", "racing thoughts"], "meds": [("Lithium", "31299"), ("Lamotrigine", "116527")]}
}

# 2. Dynamically extract everything we want the NER model to ignore
medication_allowlist = set()

for profile, data in PROFILES.items():
    # Extract medication names (e.g., "Sertraline", "Buspirone")
    for med_tuple in data["meds"]:
        medication_allowlist.add(med_tuple[0].lower())

    # Extract individual symptom words
    for symptom in data["symptoms"]:
        for word in symptom.split():
            medication_allowlist.add(word.lower())

# Add common generic metadata terms
medication_allowlist.update(["ozempic", "semaglutide", "medication", "dose", "mg"])

print(f"✅ Constants loaded and dynamic allowlist generated with {len(medication_allowlist)} clinical tokens.")

This function sends the text through the local model, finds the character positions of the flagged entities, and masks them dynamically from back-to-front (so the character index offsets don't shift as we modify the string).

In [ ]:
def redact_phi_ner(text, ner_pipe, allowlist):
    entities = ner_pipe(text)

    # Sort back-to-front to keep string indices accurate during replacement
    entities = sorted(entities, key=lambda x: x['start'], reverse=True)

    redacted_text = text
    for entity in entities:
        start = entity['start']
        end = entity['end']
        label = entity['entity_group']

        # Clean up the exact text string the model flagged
        flagged_word = text[start:end].strip().strip("*:.,()\"'").lower()

        # FIX: Check if the flagged word is equal to OR part of any word in our allowlist
        # This catches when BERT splits "Buspirone" into "Bus" and "pirone"
        if any(flagged_word in allowed_item for allowed_item in allowlist):
            continue

        # Apply standard PHI masks
        if label == "PER":
            mask = "[REDACTED_NAME]"
        elif label == "LOC":
            mask = "[REDACTED_LOCATION]"
        elif label == "ORG":
            mask = "[REDACTED_ORGANIZATION]"
        else:
            mask = f"[REDACTED_{label}]"

        redacted_text = redacted_text[:start] + mask + redacted_text[end:]

    return redacted_text

In [ ]:
# Updated test sentence containing a medication name from your PROFILES
sample_transcript = "Therapist: Welcome back. Elias, have you been taking your Buspirone consistently since moving to Mississauga? Elias: Yes, it helps."

print("--- Original ---")
print(sample_transcript)

print("\n--- Redacted Output ---")
# Testing the function with the NER pipeline and the clinical allowlist
cleaned_sample = redact_phi_ner(sample_transcript, ner_pipeline, medication_allowlist)
print(cleaned_sample)

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "mental-health-llm-pip"
bq_client = bigquery.Client(project=PROJECT_ID)

print("🔍 Fetching a small sample of real sessions from BigQuery...")

sample_query = f"""
    SELECT session_id, raw_transcript_unredacted
    FROM `{PROJECT_ID}.patient_insights.fact_sessions`
    LIMIT 5
"""

sample_df = bq_client.query(sample_query).to_dataframe()
print(f"✅ Loaded {len(sample_df)} sample sessions.")

# Apply your NER redaction function, passing the medication_allowlist here:
print("🧹 Running Hugging Face NER over the sample transcripts...")
sample_df['redacted_transcript'] = sample_df['raw_transcript_unredacted'].apply(
    lambda x: redact_phi_ner(x, ner_pipeline, medication_allowlist)
)
print("✅ Redaction processing complete.\n")

# --- VISUAL VERIFICATION LOOP ---
for index, row in sample_df.iterrows():
    print(f"=================== SAMPLE SESSION {index + 1} ===================")
    print("👉 BEFORE REDACTION (First 250 characters):")
    print(row['raw_transcript_unredacted'][:250] + "...")
    print("\n🔒 AFTER REDACTION (First 250 characters):")
    print(row['redacted_transcript'][:250] + "...")
    print("\n" + "="*50 + "\n")

In [ ]:
import re

print("🔍 Scanning full transcripts for successfully redacted tokens...\n")

for index, row in sample_df.iterrows():
    redacted_text = row['redacted_transcript']

    # Use regex to find any instance of our redaction tokens
    found_tokens = re.findall(r'\[REDACTED_\w+\]', redacted_text)

    print(f"=================== SESSION {index + 1} DIAGNOSTIC ===================")
    if found_tokens:
        print(f"✅ Found {len(found_tokens)} redacted entities: {set(found_tokens)}")

        # Pull out lines containing the redaction so we can eyeball them
        lines = redacted_text.split('\n')
        for line in lines:
            if "[REDACTED_" in line:
                # Print the cleaned line (truncated slightly if it's massive)
                print(f"   📍 Line: {line.strip()[:120]}...")
    else:
        print("ℹ️ No PHI detected in this specific session text.")
    print("="*50 + "\n")